# Exercise 1: Tensor basics 
In this exercise you will learn the basics of tensor creation, manipulation, indexing, broadcasting, vectorization, einsum, and attention masking fundamentals. These basics are important for understanding any complex implementation later on so make sure you understand them well.

**To complete this exercise fill in all TODOs in the functions below.** 

Make sure to check the output of your function and whether or not it fulfills the requirements outlined in the function definition. Do NOT change the function signature or name since we will be running checks on your functions during grading.

### Shape legend used in this notebook
- `B`: batch size
- `T`: sequence length / time
- `D`: feature dimension
- `H`: number of attention heads
- `Dh`: per-head feature dimension

### Debugging tip: what to print
When you get a shape error, print:
- `x.shape`, `x.dtype`, `x.device`
- `x.is_contiguous()` (important for `view`)
For masks also print:
- `mask.shape`, `mask.dtype`, `mask.sum()` and a small slice like `mask[0, :10]`

### Reproducibility tip: seeding in PyTorch
Many operations in deep learning involve randomness (e.g., initializing model weights, shuffling data, dropout, random augmentations).
**Seeding** sets the starting state of PyTorch’s random number generator so that these random choices become **repeatable**.

- If you set the same seed and run the same code again, you should get the same *random* tensors / initial weights.
- If you don’t set a seed, results can vary between runs.

Common usage: `torch.manual_seed(seed)`

Note: even with fixed seeds, some GPU operations can still be non-deterministic due to performance optimizations. For this assignment, seeding is mainly to make debugging easier and to ensure everyone can reproduce the same intermediate results. If you are given a seed, make sure to use it when creating tensors or performing other operations.

## Tensor creation
This warmup exercise teaches you how to create tensors with different shapes and values. A few details about tensor creation that are good to know:
- `torch.tensor([...])` infers dtype from Python values (ints → integer tensor, floats → float tensor).
- `torch.arange(start, end)` is **end-exclusive**.
- `torch.linspace(start, end, steps)` is **end-inclusive**.

In [1]:
from collections.abc import Sequence
import torch

In [2]:
def make_tensor(data, dtype: torch.dtype | None = None, device: torch.device | str | None = None) -> torch.Tensor:
    """ Create a tensor from Python data (list/tuple/nested lists). """

    tensor = torch.tensor(data, dtype=dtype, device=device)
    return tensor

x = make_tensor([[1, 2], [3, 4]], dtype=torch.float32)
print(x)

tensor([[1., 2.],
        [3., 4.]])


In [3]:
def make_zeros(shape: Sequence[int], dtype: torch.dtype | None = None, device: torch.device | str | None = None) -> torch.Tensor:
    """Create a tensor filled with zeros."""
    
    tensor = torch.zeros(shape, dtype=dtype, device=device)
    return tensor

z = make_zeros((2, 3), dtype=torch.float64)
print(z)

tensor([[0., 0., 0.],
        [0., 0., 0.]], dtype=torch.float64)


In [4]:
def make_ones_like(x: torch.Tensor) -> torch.Tensor:
    """Create a tensor of ones with the same shape, dtype, and device as x. """
    tensor=torch.ones_like(x)
    tensor_2 = torch.ones((2,3), dtype=x.dtype, device=x.device) # to see the difference
    return tensor, tensor_2

base = torch.randn(2, 3, dtype=torch.float32)
ones, ones_2 = make_ones_like(base)

print(base)
print(ones)
print(ones_2)

tensor([[-1.6057, -1.6003,  0.3719],
        [-0.1625, -1.2240, -1.4832]])
tensor([[1., 1., 1.],
        [1., 1., 1.]])
tensor([[1., 1., 1.],
        [1., 1., 1.]])


In [5]:
def make_arange(start: int, end: int, step: int = 1, dtype: torch.dtype | None = None, device: torch.device | str | None = None) -> torch.Tensor:
    """Create a 1D tensor containing values [start, start+step, ..., < end]."""
   
    tensor=torch.arange(start, end, step, dtype=dtype, device=device)
    return tensor

ar = make_arange(0, 5, 2, dtype=torch.int64)
print(ar)

tensor([0, 2, 4])


In [6]:
def make_linspace(start: float, end: float, steps: int, dtype: torch.dtype | None = None, device: torch.device | str | None = None) -> torch.Tensor:
    """Create a 1D tensor with evenly spaced values from start to end (inclusive)."""
    tensor=torch.linspace(start, end, steps, dtype=dtype, device=device)
    return tensor

ls = make_linspace(0.0, 1.0, steps=5, dtype=torch.float32)
print(ls)

tensor([0.0000, 0.2500, 0.5000, 0.7500, 1.0000])


In [7]:
from torch import seed

def make_randn(shape: Sequence[int], seed: int | None = None, dtype: torch.dtype | None = None, device: torch.device | str | None = None) -> torch.Tensor:
    """Create a tensor filled with values from a standard normal distribution."""

    tensor = torch.randn(shape, dtype=dtype, device=device)
    if seed is not None:
        torch.manual_seed(seed)
    return tensor

a = make_randn((2, 3), seed=123, dtype=torch.float32)
print(a)

tensor([[ 0.6926,  0.2911,  0.1011],
        [-0.8466,  0.5714,  1.1685]])


In [8]:
def cast_dtype_and_move(x: torch.Tensor, device: torch.device, dtype: torch.dtype) -> torch.Tensor:
    """Convert tensor dtype and move to device."""
    print(x)
    tensor = x.to(device=device, dtype=dtype)
    return tensor

casted = cast_dtype_and_move(torch.tensor([1, 2, 3]), torch.device("cpu"), torch.float32)
print(casted)

tensor([1, 2, 3])
tensor([1., 2., 3.])


## Shape manipulation
Now that we covered the basic tensor creation schemes, we want to focus on shape manipulation. Understanding the difference between these mechanisms is key for building larger systems and many people still get it wrong. 
The core ideas to understand are:
- **Contiguous tensors** store data in a single, row-major memory layout.
- Many ops (especially slicing like `x[:, ::2]`, `transpose`, `permute`) often create **non-contiguous** tensors (no copy but different strides).
- `view(...)` is **zero-copy** but typically requires **contiguous** memory → may throw an error.
- `reshape(...)` tries to return a view, but if the tensor is non-contiguous it will **allocate/copy**.
- `contiguous()` forces a contiguous copy when the tensor isn’t contiguous.

If you *need* a view after reordering dims: call `x = x.contiguous()` first (this makes a contiguous copy).

In [9]:
def reshape_tensor(x: torch.Tensor, new_shape: Sequence[int]) -> torch.Tensor:
    """Reshape tensor to new_shape (may return a view or a copy)."""
    
    tensor = x.reshape(new_shape)
    return tensor

x = torch.arange(6)
print(x)
y = reshape_tensor(x, (2, 3))
print(y)

tensor([0, 1, 2, 3, 4, 5])
tensor([[0, 1, 2],
        [3, 4, 5]])


In [10]:
def view_tensor(x: torch.Tensor, new_shape: Sequence[int]) -> torch.Tensor:
    """View tensor as new_shape (requires contiguous memory and doesn't allocate new memory for the tensor data)."""
    tensor=x.view(new_shape)
    return tensor

y_view = view_tensor(x, (2, 3))
print(x)
print(y)
print(y_view)

tensor([0, 1, 2, 3, 4, 5])
tensor([[0, 1, 2],
        [3, 4, 5]])
tensor([[0, 1, 2],
        [3, 4, 5]])


In [11]:
def flatten_from_dim(x: torch.Tensor, start_dim: int = 0) -> torch.Tensor:
    """Flatten a tensor starting from start_dim into a single dimension."""
    tensor = x.flatten(start_dim=start_dim)
    return tensor

x2 = torch.randn(2, 3, 4)
flat = flatten_from_dim(x2, start_dim=1)
print(x2)
print(x2.shape)
print(flat)
print(flat.shape)

tensor([[[ 0.3374, -0.1778, -0.3035, -0.5880],
         [ 0.3486,  0.6603, -0.2196, -0.3792],
         [-0.1606, -0.4015,  0.6957, -1.8061]],

        [[ 1.8960, -0.1750,  1.3689, -1.6033],
         [-0.7849, -1.4096, -0.4076,  0.7953],
         [ 0.9985,  0.2212,  1.8319, -0.3378]]])
torch.Size([2, 3, 4])
tensor([[ 0.3374, -0.1778, -0.3035, -0.5880,  0.3486,  0.6603, -0.2196, -0.3792,
         -0.1606, -0.4015,  0.6957, -1.8061],
        [ 1.8960, -0.1750,  1.3689, -1.6033, -0.7849, -1.4096, -0.4076,  0.7953,
          0.9985,  0.2212,  1.8319, -0.3378]])
torch.Size([2, 12])


In [12]:
def add_singleton_dim(x: torch.Tensor, dim: int) -> torch.Tensor:
    """Insert a size-1 dimension at position dim."""
    tensor = x.unsqueeze(dim)
    return tensor

x3 = torch.randn(5, 7)
x3s = add_singleton_dim(x3, dim=1)
print(x3)
print(x3s)
print(x3.shape)
print(x3s.shape)

tensor([[ 0.9624,  0.2492, -0.4845, -2.0929, -0.8199, -0.4210, -0.9620],
        [ 1.2825, -0.3430, -0.6821, -0.9887, -1.7018, -0.7498, -1.1285],
        [ 0.4135,  0.2892, -0.2044, -2.2685, -0.9133, -1.1440,  0.2436],
        [-0.0567,  0.3784,  1.6863,  0.2553, -0.5496,  1.0042,  0.3507],
        [ 1.5434,  0.1406,  1.0617, -0.9929, -1.6025, -1.0764,  0.9031]])
tensor([[[ 0.9624,  0.2492, -0.4845, -2.0929, -0.8199, -0.4210, -0.9620]],

        [[ 1.2825, -0.3430, -0.6821, -0.9887, -1.7018, -0.7498, -1.1285]],

        [[ 0.4135,  0.2892, -0.2044, -2.2685, -0.9133, -1.1440,  0.2436]],

        [[-0.0567,  0.3784,  1.6863,  0.2553, -0.5496,  1.0042,  0.3507]],

        [[ 1.5434,  0.1406,  1.0617, -0.9929, -1.6025, -1.0764,  0.9031]]])
torch.Size([5, 7])
torch.Size([5, 1, 7])


In [13]:
def remove_singleton_dims(x: torch.Tensor, dim: int | None = None) -> torch.Tensor:
    """Remove size-1 dimensions."""
    tensor = x.squeeze(dim)
    return tensor

x4 = torch.randn(2, 1, 3)
x4s = remove_singleton_dims(x4, dim=1)
print(x4)
print(x4.shape)
print(x4s)
print(x4s.shape)

tensor([[[ 1.4614,  0.8828,  0.8846]],

        [[-1.2506,  1.0758,  1.6966]]])
torch.Size([2, 1, 3])
tensor([[ 1.4614,  0.8828,  0.8846],
        [-1.2506,  1.0758,  1.6966]])
torch.Size([2, 3])


In [14]:
def transpose_last_two(x: torch.Tensor) -> torch.Tensor:
    """Swap the last two dimensions of x."""
    tensor = x.transpose(-1, 2)
    # same as tensor = x.transpose(-1, -2) or (2,1) or (-2,-1), we are swaping dimensions
    return tensor

torch.manual_seed(0) 
x6 = torch.randn(2, 3, 4)
x6t = transpose_last_two(x6)
print(x6)
print(x6.shape)
print(x6t)
print(x6t.shape)

tensor([[[-1.1258, -1.1524, -0.2506, -0.4339],
         [ 0.8487,  0.6920, -0.3160, -2.1152],
         [ 0.4681, -0.1577,  1.4437,  0.2660]],

        [[ 0.1665,  0.8744, -0.1435, -0.1116],
         [ 0.9318,  1.2590,  2.0050,  0.0537],
         [ 0.6181, -0.4128, -0.8411, -2.3160]]])
torch.Size([2, 3, 4])
tensor([[[-1.1258, -1.1524, -0.2506, -0.4339],
         [ 0.8487,  0.6920, -0.3160, -2.1152],
         [ 0.4681, -0.1577,  1.4437,  0.2660]],

        [[ 0.1665,  0.8744, -0.1435, -0.1116],
         [ 0.9318,  1.2590,  2.0050,  0.0537],
         [ 0.6181, -0.4128, -0.8411, -2.3160]]])
torch.Size([2, 3, 4])


In [15]:
def permute_bhwc_to_bchw(x: torch.Tensor) -> torch.Tensor:
    """Convert (B, H, W, C) tensor into (B, C, H, W)."""
    tensor = x.permute(0, 3, 1, 2)
    return tensor

x7 = torch.randn(8, 32, 32, 3)
x7p = permute_bhwc_to_bchw(x7)
# print(x7)
print(x7.shape)
# print(x7p)
print(x7p.shape)

torch.Size([8, 32, 32, 3])
torch.Size([8, 3, 32, 32])


In [16]:
def make_contiguous(x: torch.Tensor) -> torch.Tensor:
    """Check if tensor is contiguous and if not make contiguous."""
    if not x.is_contiguous():
        print("Tensor is not contiguous. Making it contiguous.")
        tensor = x.contiguous()
    else:
        tensor = x
    return tensor


x8 = torch.randn(4, 6)[:, ::2]
x8c = make_contiguous(x8)
print(x8)
print(x8.shape)
print(x8c)
print(x8c.shape)

Tensor is not contiguous. Making it contiguous.
tensor([[-1.7622,  0.1341,  1.6914],
        [ 0.4437,  0.7280, -1.5551],
        [-0.0819,  0.4449,  0.2066],
        [-0.4353,  0.6696,  0.8902]])
torch.Size([4, 3])
tensor([[-1.7622,  0.1341,  1.6914],
        [ 0.4437,  0.7280, -1.5551],
        [-0.0819,  0.4449,  0.2066],
        [-0.4353,  0.6696,  0.8902]])
torch.Size([4, 3])


## Indexing
Now that we know how to create tensors and manipulate them we need to understand how we can extract certain components from them using indexing. 
- Basic slicing (`x[a:b]`) returns a view when possible.
- “Fancy” indexing (lists/tensors of indices) usually allocates a new tensor.
- In-place vs out-of-place matters: if a function says “return a copy, leave the input unchanged”, you need `clone()`.

In [17]:
def slice_rows(x: torch.Tensor, start: int, end: int) -> torch.Tensor:
    """Slice rows in a 2D tensor: x[start:end, :]."""
    tensor = x[start:end, :]
    return tensor

x = torch.arange(12).reshape(4, 3)
rows = slice_rows(x, 1, 3)
print(x)
print (rows)

tensor([[ 0,  1,  2],
        [ 3,  4,  5],
        [ 6,  7,  8],
        [ 9, 10, 11]])
tensor([[3, 4, 5],
        [6, 7, 8]])


In [18]:
def select_columns(x: torch.Tensor, cols: Sequence[int]) -> torch.Tensor:
    """Select specific columns from a 2D tensor."""
    tensor = x[:, cols]
    return tensor

cols = select_columns(x, [0, 2])
print (x) 
print (cols)

tensor([[ 0,  1,  2],
        [ 3,  4,  5],
        [ 6,  7,  8],
        [ 9, 10, 11]])
tensor([[ 0,  2],
        [ 3,  5],
        [ 6,  8],
        [ 9, 11]])


In [19]:
def get_diagonal(x: torch.Tensor) -> torch.Tensor:
    """Get the diagonal of a 2D tensor."""
    return torch.diagonal(x)

d = get_diagonal(torch.tensor([[1, 2], [3, 4]]))
print (d)

tensor([1, 4])


In [20]:
def set_subtensor(x: torch.Tensor, row_idx: int, col_idx: int, value: float) -> torch.Tensor:
    """Return a copy of x where x[row_idx, col_idx] is set to value."""
    y=x.clone()
    y[row_idx, col_idx]=value
    return y

base = torch.zeros(2, 2)
out = set_subtensor(base, 0, 1, 5.0)
print(base)
print(out)

tensor([[0., 0.],
        [0., 0.]])
tensor([[0., 5.],
        [0., 0.]])


In [21]:
def gather_rows(x: torch.Tensor, row_indices: torch.Tensor) -> torch.Tensor:
    """Gather (concat) rows from x using row_indices."""
    return x[row_indices]

x2 = torch.tensor([[10, 11], [20, 21], [30, 31]])
idx = torch.tensor([2, 0])

gathered = gather_rows(x2, idx)
print (x2)

print (gathered)

tensor([[10, 11],
        [20, 21],
        [30, 31]])
tensor([[30, 31],
        [10, 11]])


## Broadcasting and reducing
Now we're covering a pytorch mechanism that lets you apply elementwise ops without using python loops. It's important to understand how it works to trace your shapes in complicated systems. The broadcasting rules to know are:
- Dimensions align from the **right**.
- A dimension can broadcast if it’s equal or one of them is **1**.

### Reduction ops and `keepdim`

When you reduce over a dimension (e.g. `sum`, `mean`, `max`), PyTorch can either:

- **remove** the reduced dimension (`keepdim=False`, default), or
- **keep** it as size 1 (`keepdim=True`)

Keeping the dimension is often helpful because it makes broadcasting back “just work”.

#### Shape diagram examples

Assume `x` has shape `(B, T, D)`:

**Sum over time**
- `x.sum(dim=1)` → shape `(B, D)`
- `x.sum(dim=1, keepdim=True)` → shape `(B, 1, D)`

**Mean over features**
- `x.mean(dim=2)` → shape `(B, T)`
- `x.mean(dim=2, keepdim=True)` → shape `(B, T, 1)`

#### Why `keepdim=True` helps with broadcasting

Example: center `x` by subtracting the mean over `T`

- If `m = x.mean(dim=1)` has shape `(B, D)`, then `x - m` **fails** (shapes `(B,T,D)` and `(B,D)` don't align).
- If `m = x.mean(dim=1, keepdim=True)` has shape `(B,1,D)`, then `x - m` **works** via broadcasting.

In [22]:
def sum_over_dim(x: torch.Tensor, dim: int, keepdim: bool = False) -> torch.Tensor:
    """Sum tensor values along dimension dim."""
    tensor = torch.sum(x, dim=dim, keepdim=keepdim)
    return tensor

x = torch.ones(2, 3)
y = sum_over_dim(x, dim=0, keepdim=True)
print(x)
print(x.shape)
print (y)
print (y.shape)

tensor([[1., 1., 1.],
        [1., 1., 1.]])
torch.Size([2, 3])
tensor([[2., 2., 2.]])
torch.Size([1, 3])


In [23]:
def mean_over_dim(x: torch.Tensor, dim: int, keepdim: bool = False) -> torch.Tensor:
    """Mean along dimension dim."""
    tensor = torch.mean(x, dim=dim, keepdim=keepdim)
    return tensor

x2 = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
y2 = mean_over_dim(x2, dim=0, keepdim=True)
print(x2)
print(x2.shape)
print (y2)
print (y2.shape)

tensor([[1., 2.],
        [3., 4.]])
torch.Size([2, 2])
tensor([[2., 3.]])
torch.Size([1, 2])


In [24]:
def max_over_dim(x: torch.Tensor, dim: int) -> tuple[torch.Tensor, torch.Tensor]:
    """Max values and argmax indices along dimension dim."""
    values, idx = torch.max(x, dim=dim, keepdim=True)
    return values, idx

x3 = torch.tensor([[1.0, 5.0], [3.0, 2.0]])
values, idx = max_over_dim(x3, dim=1)
print(x3)
print(x3.shape)
print(values)
print(values.shape)
print(idx)



tensor([[1., 5.],
        [3., 2.]])
torch.Size([2, 2])
tensor([[5.],
        [3.]])
torch.Size([2, 1])
tensor([[1],
        [0]])


In [25]:
def argmax_over_dim(x: torch.Tensor, dim: int) -> torch.Tensor:
    """Argmax indices along dimension dim."""
    tensor = torch.argmax(x, dim=dim)
    return tensor

idx2 = argmax_over_dim(x3, dim=1)
print(idx2)

tensor([1, 0])


In [26]:
def broadcast_add_vector(x: torch.Tensor, v: torch.Tensor) -> torch.Tensor:
    """Add a vector v to each row of a 2D tensor x using broadcasting."""
    y = x + v
    return y

x4 = torch.zeros(3, 2)
v = torch.tensor([10.0, 20.0])
y4 = broadcast_add_vector(x4, v)
# print(v.shape)
# print (v.unsqueeze(0).shape)
print (y4)

tensor([[10., 20.],
        [10., 20.],
        [10., 20.]])


## Vectorization
We want to avoid slow (due to per-iteration overhead) python loops as much as possible and pytorch gives us many tools to avoid it. We cover these basics:
- `cat` vs `stack` (concatenate existing dims vs create a new dim)
- `repeat` vs `expand`
- `scatter_add` / `index_add` for accumulation
- `where` for conditional selection

### `expand` vs `repeat`

- `repeat(...)` **copies** data → larger tensor with independent storage.
- `expand(...)` **does not copy** data → it creates a *view* with clever strides.

This has two important implications:

1) `expand` only works when expanding a **size-1 dimension** (broadcasting a singleton).
2) The expanded tensor may have **many positions pointing to the same memory**.  
   Modifying the expanded tensor can therefore produce surprising results (multiple rows change).

Rule of thumb:
- Use `expand` for read-only broadcasting.
- Use `repeat` if you truly need independent copies.


NOTE: We implore you to write your own quick checks from now on for calling the functions and checking their output. As before you are still required to fill in the TODOs in each function.

In [27]:
def concat_tensors(tensors: Sequence[torch.Tensor], dim: int = 0) -> torch.Tensor:
    """Concatenate tensors along dim. NOTE: This will always allocate new memory"""
    return torch.cat(tensors, dim=dim)

# example
t1 = torch.tensor([[1, 2], [3, 4]])
t2 = torch.tensor([[5, 6], [7, 8]])
result = concat_tensors([t1, t2], dim=1)
print(result)

tensor([[1, 2, 5, 6],
        [3, 4, 7, 8]])


In [28]:
def stack_tensors(tensors: Sequence[torch.Tensor], dim: int = 0) -> torch.Tensor:
    """Stack tensors along a new dimension dim."""
    return torch.stack(tensors, dim=dim)

#example
t3 = torch.tensor([[1, 2], [3, 4]])
t4 = torch.tensor([[5, 6], [7, 8]])
result2 = stack_tensors([t3, t4], dim=2)

print(result2)

tensor([[[1, 5],
         [2, 6]],

        [[3, 7],
         [4, 8]]])


In [29]:
def repeat_tensor(x: torch.Tensor, repeats: Sequence[int]) -> torch.Tensor:
    """Repeat tensor along each dimension."""
    print(repeats)
    return x.repeat(repeats)

#example
t5 = torch.tensor([[1, 2], [3, 4]])
result = t5.repeat(2, 3)  # Repeat 2 times along dim 0 and 3 times along dim 1

In [30]:
def expand_tensor(x: torch.Tensor, *sizes: int) -> torch.Tensor:
    """Expand tensor to a larger size without copying data.(Sizes can be -1 to keep original dimension.)"""
    return x.expand(*sizes)

t6 = torch.tensor([[1], [2]])
result3 = expand_tensor(t6, 2, 3)  # Expand to shap
# result3 = t6.repeat(2, 3)  # Expand to shape (2, 3)
print(result3)
    

tensor([[1, 1, 1],
        [2, 2, 2]])


In [31]:
def cumsum_over_dim(x: torch.Tensor, dim: int = 0) -> torch.Tensor:
    """Cumulative sum along dim."""
    y = torch.cumsum(x, dim=dim)
    return y

t7 = torch.tensor([[1, 2], [3, 4], [5, 6]])
result4 = cumsum_over_dim(t7, dim=0)  # Cumulative sum
print(result4)

tensor([[ 1,  2],
        [ 4,  6],
        [ 9, 12]])


In [32]:
def where_select(mask: torch.Tensor, a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    """Elementwise select: return a where mask is True else b. mask must be broadcastable to a and b."""

    tensor = torch.where(mask, a, b)
    return tensor

where_result = where_select(torch.tensor([[True, False], [False, True]]), torch.tensor([[1, 2], [3, 4]]), torch.tensor([[5, 6], [7, 8]]))

print(where_result)

tensor([[1, 6],
        [7, 4]])


In [33]:
def one_hot(indices: torch.Tensor, num_classes: int, dtype: torch.dtype | None = None) -> torch.Tensor:
    """
    Create one-hot encodings.
    Output is a tensor of the same shape as indices with an added dimension of size num_classes at the end, 
    where the value along that dimension is 1 if it matches the index and 0 otherwise.

    Shapes:
    - indices: (...,) integer tensor
    Return:
    - out: (..., num_classes)

    Requirements:
    - Must work for arbitrary leading shape.
    - No Python loops.
    """
    y_shape = indices.shape + (num_classes,)
    y = torch.zeros(y_shape, dtype=dtype)
    print (indices.unsqueeze(-1))
    y.scatter_(-1, indices.unsqueeze(-1), 1) #y.scatter_(dim, index, value)
    return y

one_hot_result = one_hot(torch.tensor([0, 2, 1]), num_classes=3)
print(one_hot_result)


tensor([[0],
        [2],
        [1]])
tensor([[1., 0., 0.],
        [0., 0., 1.],
        [0., 1., 0.]])


In [34]:
def scatter_add_1d(
    values: torch.Tensor, indices: torch.Tensor, size: int
) -> torch.Tensor:
    """
    Sum `values` into an output vector at positions `indices`.

    Shapes:
    - values: (N,)
    - indices: (N,) integer indices in [0, size)
    Return:
    - out: (size,) with same dtype and device as values

    Requirement:
    - no Python loops
    """
    out = torch.zeros(size, dtype=values.dtype, device=values.device)
    out.scatter_add_(0, indices, values)
    return out      

# example#
values = torch.tensor([1.0, 2.0, 3.0])
indices = torch.tensor([0, 1, 0])
size = 6
result5 = scatter_add_1d(values, indices, size)
print (result5)  

tensor([4., 2., 0., 0., 0., 0.])


In [35]:
def batched_token_histogram(tokens: torch.Tensor, vocab_size: int) -> torch.Tensor:
    """
    Count token occurrences per batch item.

    Shapes:
    - tokens: (B, T) int64
    Return:
    - counts: (B, vocab_size) where counts[b, v] = number of times token v appears in tokens[b] 

    Requirements:
    - No Python loops over B or T.
    """
    # y_shape = tokens.shape + (vocab_size,)
    # y=torch.zeros(y_shape, dtype=torch.int64, device=tokens.device)
    # hstgrm = y.scatter_(-1, tokens.unsqueeze(-1), 1).sum(dim=1)
    y=torch.zeros(tokens.shape , dtype=torch.int64, device=tokens.device)
    hstgrm = y.scatter_add_(1, tokens, torch.ones(tokens.shape, dtype=torch.int64))

    return hstgrm

# example
tokens_example = torch.tensor([[0, 1, 2], [1, 2, 0]])
counts_example = batched_token_histogram(tokens_example, vocab_size=3)
print(counts_example)


tensor([[1, 1, 1],
        [1, 1, 1]])


In [36]:
def masked_mean(x: torch.Tensor, mask: torch.Tensor, dim: int) -> torch.Tensor:
    """
    Mean over `dim` considering only mask==True entries.

    Convention:
    - mask: bool tensor broadcastable to x
    - mask==True means "keep this entry"

    Return: same shape as x.mean(dim=dim)

    Requirements:
    - Avoid division by zero: if all mask are False along `dim`, define mean as 0.
    """
    x_masked = x*mask
    x_masked_sum = x_masked.sum(dim=dim)
    mask_sum = mask.sum(dim=dim)
    y= torch.where(mask_sum !=0, x_masked_sum/mask_sum, torch.zeros_like(x_masked_sum))
    return y

# example
x_example = torch.tensor([[1., 2., 3.], [4., 5., 6.]])
mask_example = torch.tensor([[True, False, True], [False, True, True]])
mean_example = masked_mean(x_example, mask_example, dim=1)
print(mean_example)


tensor([2.0000, 5.5000])


## Einsum warmup
Now that you’re comfortable with shapes and broadcasting, we’ll introduce `torch.einsum`, a concise way to express tensor operations by explicitly naming axes and summing over repeated indices.

### The idea
You describe each input tensor by labeling its dimensions with letters, e.g.
- `x: (B, T, D)` → `"btd"`
- `W: (D, H)`    → `"dh"`

Then you tell einsum what output labels you want:
- `"btd,dh->bth"`

### Rules of einsum
1) **Same letter = same axis** (must match in size, except broadcastable size-1).
2) **Repeated letters are summed over** (a “contraction”).
3) **Letters that appear in the output are kept** (in that order).
4) You can **reorder axes** just by changing the output label order.

### Tiny cheat sheet
- Sum over an axis: `"btd->bt"` (sums over `d`)
- Transpose: `"ij->ji"`
- Dot product: `"d,d->"` or batched `"btd,btd->bt"`
- Matrix multiply: `"ik,kj->ij"`
- Batched matmul: `"bij,bjk->bik"`
- Outer product: `"i,j->ij"`

### How to derive an einsum (recommended workflow)
1) Write down shapes with named axes (e.g. `q: b h t d`, `k: b h s d`).
2) Decide which axes you want to **sum over** (give them the same letter in both inputs).
3) Decide which axes you want to **keep** in the output (write them after `->`).

In this section, you’ll use einsum to implement building blocks that show up in attention:
- linear projections (`x @ W`)
- dot products
- attention score matrices (`QKᵀ`)
- applying attention weights (`softmax(scores) @ V`)

NOTE: For these exercises you are required to use `torch.einsum` not `matmul` (we check). You are also not required to understand the attention mechanism at this point and the exercises are sovable without. It is good however, to remember the implementations in this exercise for future implementations.

In [37]:
def einsum_linear_btd_dh_to_bth(x: torch.Tensor, W: torch.Tensor) -> torch.Tensor:
    """
    Linear projection using einsum.

    Shapes:
    - x: (B, T, D)
    - W: (D, H)
    Return:
    - y: (B, T, H)
    """
    y = torch.einsum('btd,dh->bth', x, W)
    return y

# example
x = torch.randn(2, 3, 4)
W = torch.randn(4, 5)
y = einsum_linear_btd_dh_to_bth(x, W)
print (x)
print (W)
print(y)  


tensor([[[ 1.0694,  1.4167,  1.3085, -0.2885],
         [ 0.4203, -1.7602, -0.0641, -0.6074],
         [ 0.1569,  0.2718, -0.2133, -1.5213]],

        [[-1.8012,  0.0696, -0.2454, -1.2437],
         [ 0.8324, -0.2858,  0.9843, -1.8684],
         [ 0.5446,  0.2569, -0.3031,  0.4549]]])
tensor([[ 0.1598, -0.6676, -0.2663,  0.1933,  0.3521],
        [-0.3638,  0.0121,  1.0616,  0.2295, -0.3733],
        [ 0.3548, -0.8274,  0.6629, -0.6621,  1.6568],
        [ 0.6927, -0.6106, -1.2005,  1.7626,  0.2787]])
tensor([[[-0.0802, -1.6032,  2.4329, -0.8431,  1.9352],
         [ 0.2641,  0.1221, -1.2939, -1.3509,  0.5296],
         [-1.2033,  1.0040,  1.9317, -2.4476, -0.8236]],

        [[-1.2617,  2.1658,  1.8839, -2.3619, -1.4134],
         [-0.7079, -0.2327,  2.3705, -3.8497,  1.5100],
         [ 0.2011, -0.3874, -0.6193,  1.1667, -0.2795]]])


In [38]:
def einsum_pairwise_dot(x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    """
    Pairwise dot product between x and y.

    Shapes:
    - x: (B, T, D)
    - y: (B, T, D)
    Return:
    - dots: (B, T) where dots[b,t] = dot(x[b,t], y[b,t])
    """
    z = torch.einsum('btd,btd->bt', x, y)
    return z

In [39]:
def einsum_qk_scores(q: torch.Tensor, k: torch.Tensor) -> torch.Tensor:
    """
    Compute attention scores QK^T using einsum.

    Shapes:
    - q: (B, H, T, Dh)
    - k: (B, H, T, Dh)
    Return:
    - scores: (B, H, T, T) where scores[b,h,i,j] = dot(q[b,h,i], k[b,h,j])
    """
    x = torch.einsum('bhid,bhjd->bhij',q,k)
    return x

In [40]:
def einsum_apply_attention(weights: torch.Tensor, v: torch.Tensor) -> torch.Tensor:
    """
    Apply attention weights to values using einsum.

    Shapes:
    - weights: (B, H, T, T)
    - v:       (B, H, T, Dh)
    Return:
    - out:     (B, H, T, Dh) where out[b,h,i] = sum_j weights[b,h,i,j] * v[b,h,j]
    """
    return torch.einsum('bhij,bhjd->bhid', weights, v)

## Attention Fundamentals
This exercise introduces some building blocks of the attention mechanism which we will encounter extensively throughout the course. It's not yet required for you to fully understand the mechanism to implement the exercises. However, it's good to remember these building blocks for the future. 

To complete the exercises you should familiarize yourself with these topics:
- Stable softmax read: https://jaykmody.com/blog/stable-softmax/
- Masking: typically this means setting masked logits to -inf *before* softmax.
- For attention: causal masks are upper-triangular (no attending to the future).

In [ ]:
def stable_softmax(x: torch.Tensor, dim: int = -1) -> torch.Tensor:
    """
    Numerically stable softmax along `dim`.

    Requirements:
    - Must not overflow for large values in x.
    - Output sums to 1 along `dim`.
    """
    max_x = torch.amax(x, dim=dim, keepdim=True)
    # print (max_x, max_x.shape)
    sm_nm = torch.exp(x - max_x)
    # print (sm_nm, sm_nm.shape)
    sm_dnm = torch.sum(sm_nm, dim=dim, keepdim=True)
    # print (sm_dnm, sm_dnm.shape)
    sm = torch.divide(sm_nm, sm_dnm)
    return sm

# example that overflows without the stability trick
x = torch.tensor([[1000.0, 1001.0, 1002.0], [1003.0, 1004.0, 1005.0]])
softmax_result = stable_softmax(x)
print(softmax_result)
    

tensor([[1002.],
        [1005.]]) torch.Size([2, 1])
tensor([[0.1353, 0.3679, 1.0000],
        [0.1353, 0.3679, 1.0000]]) torch.Size([2, 3])
tensor([[1.5032],
        [1.5032]]) torch.Size([2, 1])
tensor([[0.0900, 0.2447, 0.6652],
        [0.0900, 0.2447, 0.6652]])


In [50]:
def masked_fill_tensor(x: torch.Tensor, mask: torch.Tensor, value: float) -> torch.Tensor:
    """
    Return a copy of x where positions with mask == True are replaced by `value`.
    
    Requirements:
    - mask must be broadcastable to x.
    - do NOT modify x in-place.
    """
    y = x.clone().detach()
    y = torch.where(mask, value, x)
    return y
    

In [51]:
def masked_softmax(x: torch.Tensor, mask: torch.Tensor, dim: int = -1) -> torch.Tensor:
    """
    Softmax over x with a boolean mask.

    Convention:
    - mask == True means "invalid and must receive probability 0".
    - Do masking before softmax (i.e., set invalid logits to a large negative).”

    Requirements:
    - Must be numerically stable.
    - Output must be exactly 0 where mask==True.
    - If all entries are masked along `dim`, return all zeros along `dim`.
    - You may reuse functions you implemented above.
    """
    x_masked = masked_fill_tensor (x, mask, -1e9)
    x_softmax = stable_softmax(x_masked, dim=dim)

    fully_masked = mask.all(dim=dim, keepdim=True) # True where the whole row/slice was masked
    x_softmax = torch.where(fully_masked, torch.zeros_like(x_softmax), x_softmax)

    return x_softmax
   


In [54]:
def make_causal_mask(T: int, device: torch.device | str | None = None) -> torch.Tensor:
    """
    Create a causal (future-masking) boolean mask of shape (T, T).

    Convention:
    - mask[i, j] == True  => position (i attends to j) is NOT allowed (j is in the future)
    - mask[i, j] == False => allowed

    So this is an upper-triangular mask above the diagonal.

    Return:
    - mask: boolean tensor on the specified device

    Example (T=4):
        [[F, T, T, T],
         [F, F, T, T],
         [F, F, F, T],
         [F, F, F, F]]
    """
    mask = torch.full((T, T), True, device=device)
    casual_mask = torch.triu(mask, diagonal=1) # digaonal=i keeps the the diagonal i and above it True, below it False

    return casual_mask

#example (T=4):
mask = make_causal_mask(4)
print(mask)       


tensor([[False,  True,  True,  True],
        [False, False,  True,  True],
        [False, False, False,  True],
        [False, False, False, False]])


In [56]:
def apply_causal_mask(attn_logits: torch.Tensor, value: float = -1e9) -> torch.Tensor:
    """
    Apply a causal mask to attention logits.

    Expected shapes:
    - attn_logits: (..., T, T)

    Returns:
    - masked logits (same shape) where masked positions have been set to `value`.

    Notes:
    - Create a causal mask for the final two dims.
    - Broadcast it across leading dims.
    - You may reuse functions declared above.
    """
    T = attn_logits.shape[-1]
    mask = make_causal_mask(T, attn_logits.device)
    masked_logits = masked_fill_tensor(attn_logits, mask, value)

    return masked_logits
    
    